In [3]:
import datetime
import git
import os
import itertools
import json
from collections import defaultdict
from tqdm import tqdm

react_repo_path = os.path.join(os.getcwd(), 'react')
if not os.path.exists(react_repo_path):
    print("Cloning the React repository...")
    !git clone https://github.com/facebook/react.git react

task2data_path = os.path.join(os.getcwd(), 'Task2Data')
if not os.path.exists(task2data_path):
    os.makedirs(task2data_path)

react_repo_path = os.path.join(os.getcwd(), 'react')
repo = git.Repo(react_repo_path)


In [5]:
commits = list(repo.iter_commits())

file_changes = defaultdict(list)

print("Processing commits to extract file change times...")
for commit in tqdm(commits):
    commit_time = datetime.datetime.fromtimestamp(commit.committed_date)
    files_changed = commit.stats.files.keys()
    for file in files_changed:
        file_changes[file].append(commit_time)


Processing commits to extract file change times...


100%|██████████| 19678/19678 [13:01<00:00, 25.18it/s]


In [6]:
MIN_CHANGES = 10

file_change_counts = {file: len(times) for file, times in file_changes.items()}
frequently_changed_files = [file for file, count in file_change_counts.items() if count >= MIN_CHANGES]

print(f"Total files changed more than {MIN_CHANGES} times: {len(frequently_changed_files)}")


Total files changed more than 10 times: 3005


In [7]:
time_windows = {
    '24h': datetime.timedelta(hours=24),
    '48h': datetime.timedelta(hours=48),
    '72h': datetime.timedelta(hours=72),
}


In [8]:
def count_overlaps(times1, times2, window_timedelta):
    count = 0
    j = 0
    len2 = len(times2)
    for t1 in times1:
        while j < len2 and times2[j] < t1 - window_timedelta:
            j += 1
        k = j
        while k < len2 and times2[k] <= t1 + window_timedelta:
            count += 1
            k +=1
    return count

degrees = { '24h': defaultdict(int), '48h': defaultdict(int), '72h': defaultdict(int) }

print("Calculating temporal coupling degrees...")
for file1, file2 in tqdm(itertools.combinations(frequently_changed_files, 2)):
    times1 = sorted(file_changes[file1])
    times2 = sorted(file_changes[file2])
    for window_name, window_timedelta in time_windows.items():
        count = count_overlaps(times1, times2, window_timedelta)
        if count > 0:
            degrees[window_name][(file1, file2)] = count


Calculating temporal coupling degrees...


4513510it [01:20, 55892.47it/s]


In [9]:

results = {}

for window_name in time_windows.keys():
    degree_dict = degrees[window_name]
    # Sort the degree_dict items by count descending
    sorted_pairs = sorted(degree_dict.items(), key=lambda item: item[1], reverse=True)
    top3_pairs = sorted_pairs[:3]
    results[window_name] = []
    for ((file1, file2), count) in top3_pairs:
        results[window_name].append({
            'file1': file1,
            'file2': file2,
            'degree': count
        })


In [10]:
output_file = os.path.join(task2data_path, 'temporal_coupling_results(part1).json')

with open(output_file, 'w') as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {output_file}")


Results saved to C:\Users\Francesco\PycharmProjects\FSS Software Evolution\FSS Software Evolution\Task2Data\temporal_coupling_results(part1).json
